# LangGraph Job Application Workflow

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that parses a job description, tailors a resume to match, drafts a cover letter, runs quality review, and stores revision checkpoints at every stage — so you can roll back to any prior version.

**Why this matters:** Sending the same generic resume to every job posting is the #1 reason applications get filtered out by ATS (Applicant Tracking Systems). A graph-based workflow makes resume tailoring systematic, auditable, and repeatable — and the checkpoint system means you never lose a good intermediate draft.

**Pipeline Nodes:**

```
  START
    │
    ▼
  ┌─────────────────────┐
  │  parse_job_posting   │  extract requirements, skills, keywords
  └──────────┬──────────┘
             │  checkpoint ①
             ▼
  ┌─────────────────────┐
  │  tailor_resume       │  align experience to job requirements
  └──────────┬──────────┘
             │  checkpoint ②
             ▼
  ┌─────────────────────┐
  │  draft_cover_letter  │  write personalized cover letter
  └──────────┬──────────┘
             │  checkpoint ③
             ▼
  ┌─────────────────────┐
  │  quality_review      │  score ATS alignment + quality
  └──────────┬──────────┘
        conditional
       ┌────┴────┐
       ▼         ▼
    approve    revise ──→ back to tailor_resume
       │
       ▼
  ┌─────────────────────┐
  │  compile_package     │  assemble final application package
  └──────────┬──────────┘
             │  checkpoint ④
             ▼
           END
```

**Stack:** `LangGraph` + `ChatOllama` (`qwen3.5:9b`) — fully local, no API keys

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** for a multi-stage document pipeline |
| 2 | Build **single-responsibility nodes** — parse, tailor, draft, review, compile |
| 3 | Implement **revision checkpoints** that store every intermediate version |
| 4 | Use **conditional edges** to route on quality review scores |
| 5 | Add a **revision loop** with feedback from the reviewer |
| 6 | Understand **state passing** — partial updates that accumulate through the graph |
| 7 | Compare multiple application packages across different job postings |

## 3. What Are Revision Checkpoints?

A checkpoint is a **snapshot of the state at a specific point** in the pipeline. If a later node produces a bad result, you can inspect (or roll back to) any earlier checkpoint.

```
  ┌─────────────────────────────────────────────────────────────────┐
  │                  CHECKPOINT SYSTEM                              │
  ├─────────────────────────────────────────────────────────────────┤
  │                                                                 │
  │  checkpoints = []  ← list of snapshots                         │
  │                                                                 │
  │  After parse_job_posting:                                      │
  │    checkpoints.append({                                        │
  │      "node": "parse_job_posting",                               │
  │      "timestamp": "...",                                       │
  │      "fields_written": ["job_analysis"],                       │
  │      "snapshot": {...full state...}                             │
  │    })                                                          │
  │                                                                 │
  │  After tailor_resume:                                          │
  │    checkpoints.append({...snapshot with tailored_resume...})    │
  │                                                                 │
  │  WHY: If the cover letter is bad, you can see the resume       │
  │  was fine → the problem is in draft_cover_letter, not earlier. │
  │                                                                 │
  │  HOW TO ROLL BACK:                                             │
  │    state = checkpoints[1]["snapshot"]  # back to post-resume   │
  │    result = app.invoke(state)  # re-run from that point        │
  └─────────────────────────────────────────────────────────────────┘
```

## 4. State Passing — How Partial Updates Work

```
  ┌──────────────────────────────────────────────────────────────┐
  │  After parse_job_posting:                                    │
  │    state += {job_analysis: "...requirements..."}              │
  │                                                              │
  │  After tailor_resume:                                        │
  │    state += {tailored_resume: "...customized resume..."}     │
  │    → job_analysis is STILL there, untouched                 │
  │                                                              │
  │  After draft_cover_letter:                                   │
  │    state += {cover_letter: "...personalized letter..."}      │
  │    → BOTH job_analysis AND tailored_resume preserved         │
  │                                                              │
  │  KEY: Each node returns ONLY its new fields.                 │
  │  LangGraph merges them into the cumulative state.            │
  └──────────────────────────────────────────────────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph

print("Dependencies: langchain, langgraph")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import copy
import textwrap
import warnings
from typing import TypedDict
from datetime import datetime

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print("LangGraph: StateGraph, START, END")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Input Data & State Schema

## 8. Candidate Profile (Your Resume Data)

This is the "raw material" — the candidate's full experience. The pipeline will select and reframe the most relevant parts for each job.

In [ ]:
CANDIDATE = {
    "name": "Jordan Rivera",
    "email": "jordan.rivera@email.com",
    "title": "Senior Software Engineer",
    "years_experience": 7,
    "summary": (
        "Full-stack engineer with 7 years building scalable web applications "
        "and data pipelines. Led a team of 5 engineers. Strong background in "
        "Python, TypeScript, and cloud infrastructure. Passionate about "
        "developer tools and automation."
    ),
    "skills": [
        "Python", "TypeScript", "JavaScript", "React", "Node.js",
        "FastAPI", "Django", "PostgreSQL", "Redis", "Docker",
        "Kubernetes", "AWS (EC2, S3, Lambda, RDS)", "CI/CD (GitHub Actions)",
        "GraphQL", "REST API design", "Terraform",
        "Apache Kafka", "Elasticsearch", "pandas", "scikit-learn",
    ],
    "experience": [
        {
            "role": "Senior Software Engineer",
            "company": "DataFlow Inc.",
            "dates": "2022-present",
            "bullets": [
                "Led team of 5 building real-time data pipeline processing 2M events/day using Kafka and Python",
                "Designed and deployed microservices architecture on Kubernetes, reducing deploy time by 60%",
                "Built internal developer portal with React/FastAPI, adopted by 120+ engineers",
                "Implemented CI/CD pipelines with GitHub Actions, achieving 99.5% deployment success rate",
                "Mentored 3 junior engineers through promotion to mid-level",
            ],
        },
        {
            "role": "Software Engineer",
            "company": "CloudScale Labs",
            "dates": "2019-2022",
            "bullets": [
                "Built RESTful APIs serving 50K requests/min using Django and PostgreSQL",
                "Migrated monolith to microservices, improving system reliability from 99.1% to 99.9%",
                "Created automated testing framework that reduced QA cycle from 3 days to 4 hours",
                "Optimized Elasticsearch queries, reducing search latency by 70%",
            ],
        },
        {
            "role": "Junior Developer",
            "company": "WebStart Agency",
            "dates": "2017-2019",
            "bullets": [
                "Developed 15+ client websites using React, Node.js, and MongoDB",
                "Implemented payment integrations with Stripe and PayPal APIs",
                "Built internal project management tool saving 10 hours/week per team",
            ],
        },
    ],
    "education": "B.S. Computer Science, State University (2017)",
    "certifications": ["AWS Solutions Architect Associate", "Kubernetes (CKA)"],
}

print(f"Candidate: {CANDIDATE['name']}")
print(f"  Title: {CANDIDATE['title']}")
print(f"  Experience: {CANDIDATE['years_experience']} years, {len(CANDIDATE['experience'])} roles")
print(f"  Skills: {len(CANDIDATE['skills'])} listed")

## 9. Job Postings

We'll run the pipeline on multiple job postings to show how the same candidate profile gets tailored differently.

In [ ]:
JOB_POSTINGS = [
    {
        "title": "Staff Platform Engineer",
        "company": "InfraCore Technologies",
        "location": "Remote (US)",
        "description": """
We're looking for a Staff Platform Engineer to lead our infrastructure team.
You'll design and build the developer platform that 200+ engineers rely on daily.

Responsibilities:
- Design and maintain Kubernetes-based deployment infrastructure
- Build and improve CI/CD pipelines for 40+ microservices
- Create internal developer tools and self-service portals
- Lead infrastructure architecture decisions and drive adoption
- Mentor platform team members and conduct code reviews

Requirements:
- 5+ years of software engineering experience
- Strong Kubernetes and Docker expertise
- Experience with CI/CD systems (GitHub Actions, Jenkins, or similar)
- Proficiency in Python or Go for infrastructure tooling
- AWS or GCP cloud platform experience
- Terraform or Pulumi for infrastructure as code
- Experience leading technical projects and mentoring engineers

Nice to have:
- Experience building internal developer portals
- Kafka or similar event streaming platforms
- Observability tools (Datadog, Grafana, Prometheus)
""",
    },
    {
        "title": "Senior Full-Stack Engineer",
        "company": "FinLeap Digital",
        "location": "New York, NY (Hybrid)",
        "description": """
Join our product engineering team building the next generation of
financial tools for small businesses.

Responsibilities:
- Build and maintain customer-facing web applications using React and TypeScript
- Design and implement RESTful and GraphQL APIs
- Work closely with product and design teams on user experience
- Optimize application performance and database queries
- Participate in on-call rotation and incident response

Requirements:
- 4+ years of full-stack development experience
- Strong React and TypeScript skills
- Backend experience with Node.js or Python
- PostgreSQL or similar relational database experience
- REST API design and implementation
- Experience with payment processing integrations

Nice to have:
- GraphQL experience
- Financial services or fintech background
- Redis caching experience
- Elasticsearch
""",
    },
]

print(f"Job postings: {len(JOB_POSTINGS)}")
for jp in JOB_POSTINGS:
    print(f"  {jp['title']} at {jp['company']} ({jp['location']})")

## 10. State Schema

**Field ownership — which node writes which field:**

| Field | Written By | Read By |
|-------|-----------|--------|
| `candidate_profile` | Input | tailor_resume, draft_cover_letter |
| `job_posting` | Input | parse_job_posting |
| `job_analysis` | parse_job_posting | tailor_resume, draft_cover_letter, quality_review |
| `tailored_resume` | tailor_resume | draft_cover_letter, quality_review |
| `cover_letter` | draft_cover_letter | quality_review |
| `review_result` | quality_review | routing function |
| `review_feedback` | quality_review | tailor_resume (on revision) |
| `revision_count` | quality_review | routing function |
| `final_package` | compile_package | caller |
| `checkpoints` | every node | debugging, rollback |
| `current_node` | every node | debugging |

In [ ]:
class ApplicationState(TypedDict):
    """State schema for the job application pipeline.

    The checkpoints field is a list of dicts, each containing:
      - node: which node created this checkpoint
      - timestamp: when it was created
      - fields_written: which state fields this node changed
      - snapshot: a deep copy of key state fields at that moment
    """
    candidate_profile: dict
    job_posting: dict
    job_analysis: str
    tailored_resume: str
    cover_letter: str
    review_result: str
    review_feedback: str
    revision_count: int
    final_package: str
    checkpoints: list
    current_node: str


print("State schema: ApplicationState")
for name, typ in ApplicationState.__annotations__.items():
    print(f"  {name}: {typ}")

## 11. Checkpoint Helper

Each node calls this to save a snapshot before returning.

In [ ]:
def make_checkpoint(state: dict, node_name: str, fields_written: list) -> dict:
    """Create a checkpoint snapshot of the current state.

    We deep-copy only the text fields (not the full candidate/job dicts)
    to keep checkpoints lightweight.
    """
    snapshot_fields = [
        "job_analysis", "tailored_resume", "cover_letter",
        "review_result", "review_feedback",
    ]
    snapshot = {k: state.get(k, "") for k in snapshot_fields}
    snapshot["revision_count"] = state.get("revision_count", 0)

    return {
        "node": node_name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "fields_written": fields_written,
        "snapshot": snapshot,
    }


print("Checkpoint helper defined.")
print("Each checkpoint stores: node, timestamp, fields_written, snapshot")

---

# Part B — Build the Graph Nodes

## 12. Node 1: Parse Job Posting

Extracts structured requirements from the free-text job description: required skills, nice-to-haves, key responsibilities, and ATS keywords.

**Why separate parsing from tailoring?** The same analysis feeds both the resume and cover letter. Parsing once prevents inconsistency.

In [ ]:
PARSE_SYSTEM = """You are a career coach and ATS expert. Given a job posting,
extract a structured analysis with these sections:

1. ROLE SUMMARY: one-sentence description of what they need
2. REQUIRED SKILLS: bulleted list of must-have technical skills
3. NICE-TO-HAVE SKILLS: bulleted list of bonus skills
4. KEY RESPONSIBILITIES: the 3-4 most important duties
5. ATS KEYWORDS: exact phrases from the posting that an ATS would scan for
6. CULTURE SIGNALS: what the posting reveals about team culture/values
7. SENIORITY LEVEL: junior / mid / senior / staff / principal

Be precise — quote the exact terminology from the posting."""


def parse_job_posting(state: ApplicationState) -> dict:
    """Node: Extract structured requirements from the job posting."""
    print("  [NODE] parse_job_posting")
    jp = state["job_posting"]

    analysis = ask(
        f"JOB POSTING:\nTitle: {jp['title']}\nCompany: {jp['company']}\n"
        f"Location: {jp['location']}\n\n{jp['description']}\n\n"
        f"Extract the structured analysis.",
        system=PARSE_SYSTEM,
        temperature=0.1,
    )

    # Create checkpoint
    checkpoints = list(state.get("checkpoints", []))
    new_state = {"job_analysis": analysis, "current_node": "parse_job_posting"}
    merged = {**state, **new_state}
    checkpoints.append(make_checkpoint(merged, "parse_job_posting", ["job_analysis"]))
    new_state["checkpoints"] = checkpoints

    print(f"    Analysis: {len(analysis)} chars")
    print(f"    Checkpoint #{len(checkpoints)} saved")
    return new_state


print("Node defined: parse_job_posting")
print("  READS:  job_posting")
print("  WRITES: job_analysis + checkpoint")

## 13. Node 2: Tailor Resume

Takes the candidate's full experience and the job analysis, then produces a **targeted resume** that emphasizes relevant skills and reframes bullet points to match the job's language.

If this is a revision round, it incorporates quality review feedback.

In [ ]:
TAILOR_SYSTEM = """You are a professional resume writer and ATS optimization expert.
Given a candidate's full profile and a job analysis, produce a TAILORED RESUME.

Rules:
- Reorder skills to put the most job-relevant ones first
- Rewrite bullet points to use the job posting's exact keywords
- Quantify achievements (numbers, percentages, scale)
- Omit irrelevant experience details (but keep all roles)
- Match the seniority level of the posting
- Keep to 1 page equivalent (~400 words)
- Do NOT fabricate experience — only reframe what exists

Format as a clean resume with sections:
SUMMARY | SKILLS | EXPERIENCE | EDUCATION | CERTIFICATIONS"""

TAILOR_PROMPT = """JOB ANALYSIS:
{analysis}

CANDIDATE PROFILE:
Name: {name}
Current Title: {title}
Years: {years}
Skills: {skills}
Experience:
{experience}
Education: {education}
Certifications: {certs}

{revision_section}

Write the tailored resume."""


def tailor_resume(state: ApplicationState) -> dict:
    """Node: Create a job-specific tailored resume."""
    print("  [NODE] tailor_resume")
    c = state["candidate_profile"]

    # Format experience for the prompt
    exp_text = ""
    for role in c["experience"]:
        exp_text += f"\n  {role['role']} at {role['company']} ({role['dates']})\n"
        for b in role["bullets"]:
            exp_text += f"    - {b}\n"

    revision_section = ""
    if state.get("review_feedback") and state.get("revision_count", 0) > 0:
        revision_section = (
            f"REVIEWER FEEDBACK (incorporate these improvements):\n"
            f"{state['review_feedback']}\n\n"
            f"PREVIOUS RESUME (improve, don't restart):\n"
            f"{state['tailored_resume'][:600]}"
        )
        print(f"    Revision round {state['revision_count']} — incorporating feedback")

    resume = ask(
        TAILOR_PROMPT.format(
            analysis=state["job_analysis"][:1500],
            name=c["name"],
            title=c["title"],
            years=c["years_experience"],
            skills=", ".join(c["skills"]),
            experience=exp_text,
            education=c["education"],
            certs=", ".join(c["certifications"]),
            revision_section=revision_section,
        ),
        system=TAILOR_SYSTEM,
        temperature=0.3,
    )

    checkpoints = list(state.get("checkpoints", []))
    new_state = {"tailored_resume": resume, "current_node": "tailor_resume"}
    merged = {**state, **new_state}
    checkpoints.append(make_checkpoint(merged, "tailor_resume", ["tailored_resume"]))
    new_state["checkpoints"] = checkpoints

    print(f"    Resume: {len(resume)} chars")
    print(f"    Checkpoint #{len(checkpoints)} saved")
    return new_state


print("Node defined: tailor_resume")
print("  READS:  candidate_profile, job_analysis, review_feedback (if revision)")
print("  WRITES: tailored_resume + checkpoint")

## 14. Node 3: Draft Cover Letter

Writes a cover letter that connects the candidate's experience to the specific job, referencing both the job analysis and the tailored resume for consistency.

In [ ]:
COVER_SYSTEM = """Write a professional cover letter. Rules:
- Open with a specific hook (not "I'm writing to apply for...")
- Connect 2-3 of the candidate's strongest achievements to the job's requirements
- Use the job posting's exact terminology and keywords for ATS alignment
- Show genuine interest in the company (reference their product/mission)
- Close with a confident but not arrogant call to action
- 250-350 words maximum
- Professional but personable tone
Do NOT use placeholder brackets. Write complete sentences."""

COVER_PROMPT = """JOB: {title} at {company}

JOB ANALYSIS:
{analysis}

TAILORED RESUME:
{resume}

CANDIDATE: {name}

Write the cover letter. Start with the greeting."""


def draft_cover_letter(state: ApplicationState) -> dict:
    """Node: Write a personalized cover letter."""
    print("  [NODE] draft_cover_letter")
    jp = state["job_posting"]
    c = state["candidate_profile"]

    letter = ask(
        COVER_PROMPT.format(
            title=jp["title"],
            company=jp["company"],
            analysis=state["job_analysis"][:1000],
            resume=state["tailored_resume"][:1000],
            name=c["name"],
        ),
        system=COVER_SYSTEM,
        temperature=0.4,
    )

    checkpoints = list(state.get("checkpoints", []))
    new_state = {"cover_letter": letter, "current_node": "draft_cover_letter"}
    merged = {**state, **new_state}
    checkpoints.append(make_checkpoint(merged, "draft_cover_letter", ["cover_letter"]))
    new_state["checkpoints"] = checkpoints

    print(f"    Cover letter: {len(letter)} chars")
    print(f"    Checkpoint #{len(checkpoints)} saved")
    return new_state


print("Node defined: draft_cover_letter")
print("  READS:  job_posting, candidate_profile, job_analysis, tailored_resume")
print("  WRITES: cover_letter + checkpoint")

## 15. Node 4: Quality Review

Scores the resume and cover letter on five dimensions. If the score is too low, triggers a revision loop.

**Scoring dimensions:**
- **Keyword alignment** — does the resume use the job posting's exact terms?
- **Achievement specificity** — are bullet points quantified with numbers?
- **Relevance ordering** — are the most relevant skills/experiences first?
- **Cover letter quality** — hook, body connection, CTA all present?
- **Consistency** — do the resume and cover letter tell the same story?

In [ ]:
REVIEW_SYSTEM = """You are an ATS expert and hiring manager. Score the application.
Return valid JSON only."""

REVIEW_PROMPT = """JOB ANALYSIS:
{analysis}

TAILORED RESUME:
{resume}

COVER LETTER:
{letter}

Score each dimension 1-5:
{{
  "keyword_alignment": 1-5,
  "achievement_specificity": 1-5,
  "relevance_ordering": 1-5,
  "cover_letter_quality": 1-5,
  "consistency": 1-5,
  "average_score": float,
  "decision": "approve" or "revise",
  "feedback": "specific improvements needed"
}}"""


def quality_review(state: ApplicationState) -> dict:
    """Node: Score the application package and decide approve/revise."""
    print("  [NODE] quality_review")

    raw = ask(
        REVIEW_PROMPT.format(
            analysis=state["job_analysis"][:1000],
            resume=state["tailored_resume"][:1500],
            letter=state["cover_letter"][:1000],
        ),
        system=REVIEW_SYSTEM,
        temperature=0.1,
    )
    result = parse_json(raw)

    if not result:
        result = {
            "keyword_alignment": 3, "achievement_specificity": 3,
            "relevance_ordering": 3, "cover_letter_quality": 3,
            "consistency": 3, "average_score": 3.0,
            "decision": "approve",
            "feedback": "JSON parse fell back — defaulting to approve",
        }

    dims = ["keyword_alignment", "achievement_specificity",
            "relevance_ordering", "cover_letter_quality", "consistency"]
    scores = [result.get(d, 3) for d in dims]
    avg = round(sum(scores) / len(scores), 2)
    result["average_score"] = avg

    if avg < 3.5:
        result["decision"] = "revise"
    elif avg >= 4.0:
        result["decision"] = "approve"

    revision_count = state.get("revision_count", 0)

    print(f"    Scores: {', '.join(f'{d}={result.get(d)}' for d in dims)}")
    print(f"    Average: {result['average_score']} → {result['decision'].upper()}")

    return {
        "review_result": result["decision"],
        "review_feedback": result.get("feedback", ""),
        "revision_count": revision_count + 1,
        "current_node": "quality_review",
    }


print("Node defined: quality_review")
print("  READS:  job_analysis, tailored_resume, cover_letter")
print("  WRITES: review_result, review_feedback, revision_count")

## 16. Node 5: Compile Package

Assembles the final application package with all documents and a checkpoint summary.

In [ ]:
def compile_package(state: ApplicationState) -> dict:
    """Node: Assemble final application package."""
    print("  [NODE] compile_package")
    jp = state["job_posting"]
    c = state["candidate_profile"]

    # Build checkpoint timeline
    checkpoints = state.get("checkpoints", [])
    timeline = "\n".join(
        f"  [{cp['timestamp']}] {cp['node']} → wrote {cp['fields_written']}"
        for cp in checkpoints
    )

    package = (
        f"{'=' * 60}\n"
        f"APPLICATION PACKAGE\n"
        f"{'=' * 60}\n"
        f"Candidate:  {c['name']}\n"
        f"Position:   {jp['title']} at {jp['company']}\n"
        f"Location:   {jp['location']}\n"
        f"Revisions:  {state.get('revision_count', 0)}\n"
        f"Review:     {state.get('review_result', 'N/A').upper()}\n"
        f"{'=' * 60}\n\n"
        f"--- TAILORED RESUME ---\n"
        f"{state['tailored_resume']}\n\n"
        f"--- COVER LETTER ---\n"
        f"{state['cover_letter']}\n\n"
        f"--- CHECKPOINT TIMELINE ---\n"
        f"{timeline}\n"
    )

    # Final checkpoint
    checkpoints = list(checkpoints)
    new_state = {"final_package": package, "current_node": "compile_package"}
    merged = {**state, **new_state}
    checkpoints.append(make_checkpoint(merged, "compile_package", ["final_package"]))
    new_state["checkpoints"] = checkpoints

    print(f"    Package: {len(package)} chars")
    print(f"    Total checkpoints: {len(checkpoints)}")
    return new_state


print("Node defined: compile_package")
print("  READS:  all state fields")
print("  WRITES: final_package + checkpoint")

---

# Part C — Conditional Routing & Graph Assembly

## 17. Routing After Quality Review

In [ ]:
MAX_REVISIONS = 3


def route_after_review(state: ApplicationState) -> str:
    """Conditional edge: route based on quality review."""
    decision = state.get("review_result", "approve")
    revisions = state.get("revision_count", 0)

    if decision == "approve":
        print(f"    [ROUTE] approved → compile_package")
        return "compile_package"

    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] max revisions ({MAX_REVISIONS}) → forcing compile_package")
        return "compile_package"

    print(f"    [ROUTE] revise → tailor_resume (round {revisions})")
    return "tailor_resume"


print(f"Routing: route_after_review (max {MAX_REVISIONS} revisions)")
print("  approve         → compile_package")
print("  revise (ok)     → tailor_resume (loop)")
print("  revise (at cap) → compile_package")

## 18. Assemble the StateGraph

In [ ]:
workflow = StateGraph(ApplicationState)

# Register nodes
workflow.add_node("parse_job_posting", parse_job_posting)
workflow.add_node("tailor_resume", tailor_resume)
workflow.add_node("draft_cover_letter", draft_cover_letter)
workflow.add_node("quality_review", quality_review)
workflow.add_node("compile_package", compile_package)

# Linear flow: START → parse → tailor → draft → review
workflow.add_edge(START, "parse_job_posting")
workflow.add_edge("parse_job_posting", "tailor_resume")
workflow.add_edge("tailor_resume", "draft_cover_letter")
workflow.add_edge("draft_cover_letter", "quality_review")

# Conditional edge: review routes to compile or back to tailor
workflow.add_conditional_edges(
    "quality_review",
    route_after_review,
    {
        "compile_package": "compile_package",
        "tailor_resume": "tailor_resume",
    },
)

# Terminal
workflow.add_edge("compile_package", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("Flow:")
print("  START → parse_job_posting → tailor_resume → draft_cover_letter → quality_review")
print("  quality_review ──(approve)──→ compile_package → END")
print("  quality_review ──(revise)───→ tailor_resume (loop)")

In [ ]:
# Visualize graph
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> parse_job_posting --> tailor_resume")
    print("  tailor_resume --> draft_cover_letter --> quality_review")
    print("  quality_review --(approve)--> compile_package --> __end__")
    print("  quality_review --(revise)--> tailor_resume")

---

# Part D — Execute the Pipeline

## 19. Run on Job 1 — Staff Platform Engineer

In [ ]:
def make_initial_state(candidate: dict, job: dict) -> ApplicationState:
    return {
        "candidate_profile": candidate,
        "job_posting": job,
        "job_analysis": "",
        "tailored_resume": "",
        "cover_letter": "",
        "review_result": "",
        "review_feedback": "",
        "revision_count": 0,
        "final_package": "",
        "checkpoints": [],
        "current_node": "start",
    }


print(f"Running pipeline: {JOB_POSTINGS[0]['title']} at {JOB_POSTINGS[0]['company']}")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(CANDIDATE, JOB_POSTINGS[0]),
    {"recursion_limit": 25},
)

print(f"\nPipeline complete.")
print(f"  Review: {result_1['review_result']}")
print(f"  Revisions: {result_1['revision_count']}")
print(f"  Checkpoints: {len(result_1['checkpoints'])}")
print(f"  Package: {len(result_1['final_package'])} chars")

## 20. Inspect Intermediate State

In [ ]:
print("STATE INSPECTION — Job 1")
print("=" * 80)

print("\n[1] JOB ANALYSIS (first 500 chars):")
wrap_print(result_1["job_analysis"][:500])

print("\n" + "-" * 80)
print("[2] TAILORED RESUME (first 600 chars):")
wrap_print(result_1["tailored_resume"][:600])

print("\n" + "-" * 80)
print("[3] COVER LETTER (first 500 chars):")
wrap_print(result_1["cover_letter"][:500])

print("\n" + "-" * 80)
print(f"[4] REVIEW: {result_1['review_result'].upper()}")
if result_1["review_feedback"]:
    print(f"    Feedback: {result_1['review_feedback'][:150]}")

## 21. Explore Checkpoints

Each checkpoint is a snapshot. Let's inspect what was stored at each stage.

In [ ]:
print("CHECKPOINT TIMELINE — Job 1")
print("=" * 70)

for i, cp in enumerate(result_1["checkpoints"], 1):
    print(f"\n  Checkpoint #{i}")
    print(f"    Node:          {cp['node']}")
    print(f"    Timestamp:     {cp['timestamp']}")
    print(f"    Fields written: {cp['fields_written']}")

    # Show what was in the snapshot at that point
    snap = cp["snapshot"]
    filled = {k: len(v) if isinstance(v, str) else v
              for k, v in snap.items() if v}
    print(f"    Snapshot state: {filled}")

print(f"\nTotal checkpoints: {len(result_1['checkpoints'])}")
print("Each one is a rollback point — you can restore state to any checkpoint.")

## 22. How to Roll Back to a Checkpoint

If you wanted to re-run from checkpoint #2 (after the resume was tailored), you'd restore that snapshot and re-invoke the graph.

In [ ]:
# Demonstrate rollback (conceptual)
print("ROLLBACK DEMONSTRATION")
print("=" * 60)

# Pick checkpoint #2 (after tailor_resume)
if len(result_1["checkpoints"]) >= 2:
    cp2 = result_1["checkpoints"][1]  # 0-indexed
    print(f"Rolling back to checkpoint #{2}: {cp2['node']}")
    print(f"  Timestamp: {cp2['timestamp']}")
    print(f"  At this point, the state had:")
    for k, v in cp2["snapshot"].items():
        if isinstance(v, str) and v:
            print(f"    {k}: {len(v)} chars ✓")
        elif isinstance(v, str) and not v:
            print(f"    {k}: (empty — not yet written)")
        else:
            print(f"    {k}: {v}")

    print("\n  To re-run from this point:")
    print("    restored_state = make_initial_state(CANDIDATE, JOB_POSTINGS[0])")
    print("    restored_state.update(cp2['snapshot'])")
    print("    # Then re-invoke from draft_cover_letter onward")
    print("    # (Would need a sub-graph or manual node calls)")
else:
    print("Not enough checkpoints for rollback demo.")

## 23. Streaming Step View

In [ ]:
print("STREAMING EXECUTION — Job 1")
print("=" * 60)

step = 0
for chunk in app.stream(
    make_initial_state(CANDIDATE, JOB_POSTINGS[0]),
    {"recursion_limit": 25},
):
    step += 1
    for node_name, node_output in chunk.items():
        keys = [k for k in node_output if k not in ("current_node", "checkpoints")]
        details = []
        for k in keys:
            v = node_output[k]
            if isinstance(v, str) and len(v) > 50:
                details.append(f"{k}: {len(v)} chars")
            else:
                details.append(f"{k}: {v}")
        n_cp = len(node_output.get("checkpoints", []))
        print(f"  Step {step}: {node_name:<22} → {', '.join(details)} (cp: {n_cp})")

print(f"\nTotal steps: {step}")

---

## 24. Run on Job 2 — Senior Full-Stack Engineer

Same candidate, different job. Watch how the pipeline generates completely different output.

In [ ]:
print(f"Running pipeline: {JOB_POSTINGS[1]['title']} at {JOB_POSTINGS[1]['company']}")
print("=" * 70)

result_2 = app.invoke(
    make_initial_state(CANDIDATE, JOB_POSTINGS[1]),
    {"recursion_limit": 25},
)

print(f"\nPipeline complete.")
print(f"  Review: {result_2['review_result']}")
print(f"  Revisions: {result_2['revision_count']}")
print(f"  Checkpoints: {len(result_2['checkpoints'])}")
print(f"  Package: {len(result_2['final_package'])} chars")

---

# Part E — Cross-Job Comparison

## 25. View Both Packages

In [ ]:
all_results = [result_1, result_2]

for i, (result, jp) in enumerate(zip(all_results, JOB_POSTINGS), 1):
    print(f"\n{'#' * 80}")
    print(f"# JOB {i}: {jp['title']} at {jp['company']}")
    print(f"{'#' * 80}")
    wrap_print(result["final_package"])
    print()

## 26. Pipeline Metrics

In [ ]:
print("PIPELINE METRICS")
print("=" * 80)
print(f"{'Job':<35} {'Review':<10} {'Revisions':<12} {'Checkpoints':<13} {'Package'}")
print("-" * 80)

for r, jp in zip(all_results, JOB_POSTINGS):
    label = f"{jp['title'][:30]} @ {jp['company'][:10]}"
    print(f"  {label:<33} {r['review_result']:<10} {r['revision_count']:<12} "
          f"{len(r['checkpoints']):<13} {len(r['final_package'])} chars")

avg_rev = sum(r["revision_count"] for r in all_results) / len(all_results)
avg_cp = sum(len(r["checkpoints"]) for r in all_results) / len(all_results)
print(f"\n  Avg revisions:    {avg_rev:.1f}")
print(f"  Avg checkpoints:  {avg_cp:.1f}")

## 27. Compare How the Resume Was Tailored Differently

The same candidate, two different emphases.

In [ ]:
print("RESUME TAILORING COMPARISON")
print("=" * 80)

for i, (r, jp) in enumerate(zip(all_results, JOB_POSTINGS), 1):
    print(f"\n--- JOB {i}: {jp['title']} ---")
    # Show first ~300 chars of the resume (typically the summary + top skills)
    resume_start = r["tailored_resume"][:400]
    wrap_print(resume_start)
    print(f"  ... ({len(r['tailored_resume'])} total chars)")

print("\n" + "=" * 80)
print("OBSERVATION: Same candidate, different emphasis.")
print("  Job 1 (Platform): Kubernetes, CI/CD, Terraform, infrastructure")
print("  Job 2 (Full-Stack): React, TypeScript, APIs, payments")

## 28. State Accumulation Visualization

In [ ]:
print("STATE ACCUMULATION — Job 1")
print("=" * 60)

r = all_results[0]
fields = [
    ("job_analysis", r["job_analysis"]),
    ("tailored_resume", r["tailored_resume"]),
    ("cover_letter", r["cover_letter"]),
    ("review_feedback", r["review_feedback"]),
    ("final_package", r["final_package"]),
]

cumulative = 0
for name, content in fields:
    size = len(content)
    cumulative += size
    bar = "█" * min(size // 50, 40)
    print(f"  {name:<22} {size:>5} chars  {bar}")

print(f"\n  Total text state:  {cumulative:,} chars")
print(f"  Checkpoints stored: {len(r['checkpoints'])}")
print(f"  Each checkpoint is a rollback point.")

---

## 29. Node Design Recap

```
  Node                 READS                          WRITES                  LLM?
  ──────────────────── ────────────────────────────── ─────────────────────── ─────
  parse_job_posting    job_posting                    job_analysis            Yes
  tailor_resume        candidate_profile, job_analysis, tailored_resume      Yes
                       review_feedback (opt.)
  draft_cover_letter   job_posting, candidate_profile, cover_letter          Yes
                       job_analysis, tailored_resume
  quality_review       job_analysis, tailored_resume, review_result,         Yes
                       cover_letter                    review_feedback,
                                                      revision_count
  compile_package      ALL fields                     final_package           No

  EVERY node (except quality_review) saves a checkpoint.
  quality_review doesn't need one — it's the decision point, not a document.
```

## 30. Production Extensions

1. **ATS keyword scanner** — add a node that counts exact keyword matches between resume and posting
2. **Multiple resume versions** — generate 2-3 tailored variants, let the reviewer pick the strongest
3. **LinkedIn profile optimization** — add a node that suggests LinkedIn headline/summary updates
4. **Job board integration** — pull postings from LinkedIn/Indeed APIs, run pipeline in batch
5. **Interview prep** — after approval, generate likely interview questions based on the job analysis

## 31. Exercises

### Exercise 1: Interactive Review
Replace the LLM quality review with `input()` — let yourself review the resume and cover letter, type feedback, and trigger revisions interactively.

### Exercise 2: Skill Gap Analysis
Add a `skill_gap_analysis` node between `parse_job_posting` and `tailor_resume`. It should compare required skills against the candidate's skills and flag gaps. The resume tailor should acknowledge gaps honestly rather than hiding them.

### Exercise 3: Checkpoint Diff
Write a function that compares two checkpoints and shows what changed between them — like a `git diff` for your application state. This is especially useful for seeing what the revision loop improved.

### Mini Challenge: Multi-Candidate
Extend the pipeline to accept multiple candidate profiles and rank them for a single job posting. Add a `rank_candidates` node that scores each tailored application and produces a ranked shortlist.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Candidate:        {CANDIDATE['name']}")
print(f"Jobs processed:   {len(all_results)}")
print(f"Avg revisions:    {avg_rev:.1f}")
print(f"Avg checkpoints:  {avg_cp:.1f}")
print()
print("Graph topology:")
print("  START → parse_job_posting → tailor_resume → draft_cover_letter")
print("  → quality_review ──(approve)──→ compile_package → END")
print("  → quality_review ──(revise)───→ tailor_resume (loop)")
print()
print("Nodes built:")
print("  1. parse_job_posting   — extracts requirements, keywords, and culture signals")
print("  2. tailor_resume       — rewrites resume to match job language")
print("  3. draft_cover_letter  — writes personalized letter from analysis + resume")
print("  4. quality_review      — scores on 5 ATS dimensions")
print("  5. compile_package     — assembles resume + letter + checkpoint timeline")
print()
print("Key patterns:")
print("  • Revision checkpoints at every stage for rollback")
print("  • Conditional routing based on quality scores")
print("  • Revision loop with feedback from reviewer to tailor node")
print("  • Same candidate profile produces different output per job")
print("  • Partial state updates — each node returns only its fields")
print("  • Checkpoint timeline provides full audit trail")

## 32. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Parse the job posting first** — structured requirements feed consistent downstream outputs |
| 2 | **Tailoring means reframing, not fabricating** — reorder and reword existing experience |
| 3 | **Use the job's exact keywords** — ATS systems scan for literal matches |
| 4 | **Checkpoints are snapshots, not logs** — they store full state, enabling rollback |
| 5 | **The revision loop improves quality** — reviewer feedback makes the 2nd draft measurably better |
| 6 | **Cap revision loops** — `MAX_REVISIONS` prevents infinite cycling |
| 7 | **Same candidate, different emphasis** — platform vs. full-stack roles get different skill ordering |
| 8 | **Cover letter reads the resume** — ensures consistency between documents |
| 9 | **State accumulates through the graph** — each node reads prior outputs without re-computing |
| 10 | **The graph IS the workflow** — explicit, testable, auditable, and resumable |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*